# 🚀 SOTA E-Commerce Visual Search Engine (Colab Edition)

Notebook ini akan mengunduh repositori arsitektur mesin pencari canggih (FastAPI + Qdrant + CLIP + React) yang telah kita bangun, mengonfigurasi environment di Google Colab, menjalankan server backend & frontend di latar belakang, dan memberikan Anda URL publik melalui **Localtunnel** agar Anda bisa langsung menguji purwarupa UI aplikasi ini.

---

## Langkah 1: Persiapan Environment & Download Source Code

In [1]:
!rm -rf Smart-Catalog-Amazon-Berkeley-Objects
!git clone https://github.com/Brian071/Smart-Catalog-Amazon-Berkeley-Objects.git


# Membuat direktori dataset utama
!mkdir -p dataset

# Mengunduh dataset Listings dan Images langsung dari server AWS S3
!wget -q https://amazon-berkeley-objects.s3.amazonaws.com/archives/abo-listings.tar -O dataset/abo-listings.tar
!wget -q https://amazon-berkeley-objects.s3.amazonaws.com/archives/abo-images-small.tar -O dataset/abo-images-small.tar

# Mengekstrak isi dataset ke dalam folder penampung
!tar -xf dataset/abo-listings.tar -C dataset
!tar -xf dataset/abo-images-small.tar -C dataset

# Uninstall paket yang berpotensi konflik dari memori Colab
!pip uninstall -y transformers torch torchvision numpy scipy pandas seaborn

# Install seluruh dependensi
!pip install numpy scipy pandas seaborn fastapi uvicorn torch torchvision transformers[torch] qdrant-client>=1.10.0 Pillow opencv-python optuna segment-anything python-multipart urllib3>=2 matplotlib

# Install Localtunnel
!npm install -g localtunnel

Cloning into 'Smart-Catalog-Amazon-Berkeley-Objects'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 51 (delta 10), reused 37 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 244.57 KiB | 2.16 MiB/s, done.
Resolving deltas: 100% (10/10), done.
Found existing installation: transformers 5.12.1
Uninstalling transformers-5.12.1:
  Successfully uninstalled transformers-5.12.1
Found existing installation: torch 2.12.1
Uninstalling torch-2.12.1:
  Successfully uninstalled torch-2.12.1
Found existing installation: torchvision 0.27.1
Uninstalling torchvision-0.27.1:
  Successfully uninstalled torchvision-0.27.1
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: scipy 1.17.1
Uninstalling scipy-1.17.1:
  Successfully uninstalled scipy-1.17.1
Found existing installation: pandas 3.0.3
Uninstall

## Langkah 2: Memulai Server Backend (FastAPI)

In [2]:
import os
import subprocess
import time
import requests

# Kill any existing processes on port 8000 that might be lingering from previous runs
!fuser -k 8000/tcp || true

# Jalankan Backend di background
# Temporarily remove stdout/stderr pipes to see output directly for debugging
backend_process = subprocess.Popen(
    ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/Smart-Catalog-Amazon-Berkeley-Objects/backend"
    # stdout=subprocess.PIPE,  # Removed for debugging
    # stderr=subprocess.PIPE   # Removed for debugging
)
print("Memulai Backend FastAPI di port 8000...")

# Add a robust health check
backend_ready = False
max_retries = 60 # Try for up to 60 * 1 second = 60 seconds (increased from 30)
for i in range(max_retries):
    try:
        response = requests.get("http://localhost:8000/", timeout=1)
        if response.status_code == 200:
            print("Backend FastAPI siap!")
            backend_ready = True
            break
    except requests.exceptions.ConnectionError:
        pass # Server not ready yet
    print(f"Waiting for backend... ({i+1}/{max_retries})")
    time.sleep(1)

if not backend_ready:
    print("\n=======================================================")
    print("Gagal memulai Backend FastAPI. Harap periksa log server untuk kesalahan.")
    # If pipes are not used, no need to communicate. The errors should have been printed directly by uvicorn.
    print("Check the output above for any Uvicorn/FastAPI errors.")
    print("=======================================================\n")
    backend_process.kill() # Ensure it's stopped if it failed
    raise RuntimeError("Backend FastAPI did not start within the expected time.")

Memulai Backend FastAPI di port 8000...
Waiting for backend... (1/60)
Waiting for backend... (2/60)
Waiting for backend... (3/60)
Waiting for backend... (4/60)
Waiting for backend... (5/60)
Waiting for backend... (6/60)
Waiting for backend... (7/60)
Waiting for backend... (8/60)
Waiting for backend... (9/60)
Waiting for backend... (10/60)
Waiting for backend... (11/60)
Waiting for backend... (12/60)
Waiting for backend... (13/60)
Waiting for backend... (14/60)
Waiting for backend... (15/60)
Waiting for backend... (16/60)
Waiting for backend... (17/60)
Waiting for backend... (18/60)
Waiting for backend... (19/60)
Waiting for backend... (20/60)
Waiting for backend... (21/60)
Waiting for backend... (22/60)
Waiting for backend... (23/60)
Backend FastAPI siap!


## Langkah 3: Mengindeks Dataset ABO untuk Testing
Mari kita muat beberapa sampel dataset Amazon Berkeley Objects (ABO) ke dalam Qdrant agar bisa langsung dicari.

In [3]:
import requests
from PIL import Image
import io
import json
import os
import gzip
import shutil

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# --- Step 1 & 2: Load and prepare metadata using the user's provided logic ---
print("Memuat metadata produk...")
data = []
listing_path = 'dataset/listings/listings/metadata/listings_0.json.gz'

if not os.path.exists(listing_path):
    listing_path = 'dataset/listings/metadata/listings_0.json.gz'

if not os.path.exists(listing_path):
    print(f"Warning: Listing metadata file not found at {listing_path}. Please ensure 'dataset' directory is correctly set up.")
    listing_path = None

try:
    if listing_path:
        with gzip.open(listing_path, 'rt') as f:
            for i, line in enumerate(f):
                if i >= 30000:
                    break
                try:
                    item = json.loads(line.strip())
                    data.append(item)
                except json.JSONDecodeError as e:
                    continue
except FileNotFoundError:
    print(f"Error: File tidak ditemukan di {listing_path}. Pastikan proses download dan ekstrak di cell sebelumnya selesai.")
except Exception as e:
    print(f"An error occurred while loading listings metadata: {e}")

df_listings = pd.DataFrame(data)

def extract_en_text(item_list):
    if isinstance(item_list, list):
        for entry in item_list:
            if isinstance(entry, dict) and entry.get('language_tag', '').startswith('en_'):
                return entry.get('value', '')
        if len(item_list) > 0 and isinstance(item_list[0], dict):
            return item_list[0].get('value', '')
    return None

if not df_listings.empty:
    df_listings['item_name_en'] = df_listings['item_name'].apply(extract_en_text)
    if 'product_type' in df_listings.columns:
        df_listings['category'] = df_listings['product_type'].apply(lambda x: x[0]['value'] if isinstance(x, list) and len(x) > 0 else None)
    else:
        df_listings['category'] = None

print("Memuat metadata gambar...")
images_csv_path = 'dataset/images/metadata/images.csv.gz'
if not os.path.exists(images_csv_path):
    print(f"Warning: Image metadata file not found at {images_csv_path}. Please ensure 'dataset' directory is correctly set up.")
    df_images = pd.DataFrame()
else:
    try:
        df_images = pd.read_csv(images_csv_path)
    except Exception as e:
        print(f"Error loading images.csv.gz: {e}. Creating empty DataFrame.")
        df_images = pd.DataFrame()


df_merged = pd.DataFrame()
if not df_listings.empty and not df_images.empty:
    print("Menggabungkan data produk dan gambar...")
    df_merged = pd.merge(df_listings, df_images, left_on='main_image_id', right_on='image_id', how='inner')

    if 'path' in df_merged.columns:
        df_merged['image_path'] = 'dataset/images/' + df_merged['path']
    else:
        df_merged['image_path'] = None

    df_final = df_merged[['item_id', 'image_path', 'item_name_en', 'category']].rename(columns={
        'item_id': 'id',
        'item_name_en': 'text'
    })
else:
    print("Tidak dapat menggabungkan karena satu atau lebih DataFrame kosong.")
    df_final = pd.DataFrame()

if not df_final.empty:
    print("Preview data yang akan diindeks:")
    print(df_final.head())
else:
    print("df_final kosong, tidak ada data untuk diindeks.")


## --- Step 3: Merge, Filter for shoe items, and Index the data ---
df_merged = pd.DataFrame()
if not df_listings.empty and not df_images.empty:
    print("Menggabungkan data produk dan gambar...")
    df_merged = pd.merge(df_listings, df_images, left_on='main_image_id', right_on='image_id', how='inner')

    # PERBAIKAN: Menambahkan folder 'small/' agar sesuai dengan struktur asli dataset
    if 'path' in df_merged.columns:
        df_merged['image_path'] = 'dataset/images/small/' + df_merged['path']
    else:
        df_merged['image_path'] = None

    df_final = df_merged[['item_id', 'image_path', 'item_name_en', 'category']].rename(columns={
        'item_id': 'id',
        'item_name_en': 'text'
    })
else:
    print("Tidak dapat menggabungkan karena satu atau lebih DataFrame kosong.")
    df_final = pd.DataFrame()

print("\nMulai memfilter dan mengindeks gambar sepatu ke Vector DB...")
shoe_items_df = pd.DataFrame()
if not df_final.empty and 'category' in df_final.columns:
    shoe_items_df = df_final[
        (df_final['category'].str.contains('shoes', case=False, na=False)) |
        (df_final['category'].str.contains('footwear', case=False, na=False))
    ].copy()

if shoe_items_df.empty:
    print("Tidak ditemukan item sepatu setelah filtering.")
    total_items_to_process = 0
else:
    print(f"Ditemukan {len(shoe_items_df)} item sepatu untuk diindeks.")
    total_items_to_process = len(shoe_items_df)

indexed_count = 0
failed_count = 0

if total_items_to_process > 0:
    for i, row in shoe_items_df.iterrows():
        img_id = str(row['id'])
        image_local_path = row['image_path']

        if (i + 1) % 100 == 0 or i == 0 or (i + 1) == total_items_to_process:
            print(f"Processing item {i+1}/{total_items_to_process}: ID={img_id}")

        try:
            if not os.path.exists(image_local_path):
                print(f"Error: Gambar tidak ditemukan di {image_local_path}")
                failed_count += 1
                continue

            with open(image_local_path, 'rb') as f:
                img_bytes_content = f.read()
            img_bytes = io.BytesIO(img_bytes_content)

            files = {'file': (f'{img_id}.jpg', img_bytes, 'image/jpeg')}

            # PERBAIKAN: Hanya mengirim 'id' agar sesuai dengan kebutuhan Form di FastAPI
            payload_data = {'id': img_id}

            res = requests.post("http://localhost:8000/index_image", files=files, data=payload_data)
            res.raise_for_status()
            indexed_count += 1

        except requests.exceptions.RequestException as req_err:
            print(f"Gagal memposting {img_id}: {req_err}")
            if hasattr(req_err, 'response') and req_err.response is not None:
                print(f"Detail: {req_err.response.text}")
            failed_count += 1
        except Exception as e:
            print(f"Gagal memproses {img_id}: {e}")
            failed_count += 1
else:
    print("Tidak ada data untuk diindeks.")

print(f"\n--- Indeksasi Selesai ---")
print(f"Berhasil diindeks: {indexed_count}")
print(f"Gagal diindeks: {failed_count}")

Memuat metadata produk...
Memuat metadata gambar...
Menggabungkan data produk dan gambar...
Preview data yang akan diindeks:
           id                      image_path  \
0  B06X9STHNG  dataset/images/8c/8ccb5859.jpg   
1  B07P8ML82R  dataset/images/9f/9f76d27b.jpg   
2  B07H9GMYXS  dataset/images/66/665cc994.jpg   
3  B07CTPR73M  dataset/images/b4/b4f9d0cc.jpg   
4  B01MTEI8M6  dataset/images/2b/2b1c2516.jpg   

                                                text               category  
0  Amazon-merk - vinden. Dames Leder Gesloten Tee...                  SHOES  
1  22" Bottom Mount Drawer Slides, White Powder C...               HARDWARE  
2  AmazonBasics PETG 3D Printer Filament, 1.75mm,...  MECHANICAL_COMPONENTS  
3       Stone & Beam Stone Brown Swatch, 25020039-01                   SOFA  
4  The Fix Amazon Brand Women's French Floral Emb...                  SHOES  
Menggabungkan data produk dan gambar...

Mulai memfilter dan mengindeks gambar sepatu ke Vector DB...
Ditemukan 

## Langkah 4: Menjalankan Frontend React & Ekspos ke Internet

Karena arsitektur React tidak dirancang untuk ditenagai langsung melalui Cell IPython, kita akan menjalankan proses Node JS di latar belakang dan mempublikasikan URL-nya.

**Penting:** Klik link Localtunnel yang muncul pada output di bawah untuk mengakses aplikasi!

In [ ]:
# Install module Frontend
!cd /content/Smart-Catalog-Amazon-Berkeley-Objects/frontend && npm install

# Build Frontend (Pastikan file App.tsx sudah diisi dengan URL Cloudflare Backend port 8000)
!cd /content/Smart-Catalog-Amazon-Berkeley-Objects/frontend && npm run build

# Install server statis sederhana Python dan jalankan di background
import subprocess
frontend_process = subprocess.Popen(
    ["python3", "-m", "http.server", "3000"],
    cwd="/content/Smart-Catalog-Amazon-Berkeley-Objects/frontend/build"
)
print("Frontend berjalan di port 3000...")

# Membuka jalur Cloudflare kedua khusus untuk Frontend
print("\n=======================================================")
print("MENDAPATKAN PUBLIC URL UNTUK FRONTEND DARI CLOUDFLARE...")
print("Klik tautan berakhiran .trycloudflare.com di bawah ini.")
print("=======================================================\n")
!cloudflared tunnel --url http://localhost:3000

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
up to date, audited 1322 packages in 6s
⠼
⠼272 packages are looking for funding
⠼  run `npm fund` for details
⠼
28 vulnerabilities (9 low, 6 moderate, 13 high)

To address issues that do not require attention, run:
  npm audit fix

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
⠴
> frontend@0.1.0 build
> react-scripts build

Creating an optimized production build...
Compiled successfully.

File sizes after gzip:

  62.95 kB  build/static/js/main.e4ddfc6c.js
  1.76 kB   build/static/js/453.20359781.chunk.js
  513 B     build/static/css/main.f855e6bc.css

The project was built assuming it is hosted at /.
You can control this with the homepage field in your package.json.

The build folder is ready to be deployed.
You may serve it with a static server:

  npm install -g serve
  serve -s build

Find out more about deployment here:

  https://cra.link/deployment

⠙Frontend be